In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ಹಂತ 1: ರಚಿಸಲಾದ ಔಟ್‌ಪುಟ್‌ಗಳಿಗೆ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟ್ಸ್ ಹಿಂತಿರುಗಿಸುವ **ಸ್ಕೀಮಾ** ಅನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತವೆ. ಪೈಡ್ಯಾಂಟಿಕ್‌ನ್ನು `response_format` ಜೊತೆಗೆ ಬಳಸುವುದು ಖಚಿತಪಡಿಸುತ್ತದೆ:
- ✅ ಪ್ರಕಾರ-ಸುರಕ್ಷಿತ ಡೇಟಾ ವಿಲೇವಾರಿ
- ✅ ಸ್ವಯಂಚಾಲಿತ ಮಾನ್ಯತೆ
- ✅ ಮಫಾದಿ-ಪಠ್ಯ ಉತ್ತರಗಳಿಂದ ವಿಧಿವಿನ್ಯಾಸ ದೋಷಗಳು ಇಲ್ಲ
- ✅ ಕ್ಷೇತ್ರಗಳ ಆಧಾರದ ಮೇಲೆ ಸುಲಭ ಶರತಿ ನಾವಿಗೇಶನ್


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ಹಂತ 2: ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್ ಟೂಲ್ ಸೃಷ್ಟಿಸಿ

ಈ ಸಾಧನವನ್ನು **availability_agent** ಕೋಣೆಗಳು ಲಭ್ಯವಿರುವುದನ್ನು ಪರಿಶೀಲಿಸಲು ಕರೆಮಾಡುತ್ತದೆ. ನಾವು `@ai_function` ಡೆಕೊರೇಟರ್ ಅನ್ನು ಬಳಸುತ್ತೇವೆ:
- ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಅನ್ನು AI-ಕರೆಗೆ ಸಮರ್ಥ ಸಾಧನವಾಗಿ ಪರಿವರ್ತಿಸಲು
- LLM ಗಾಗಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿ JSON ಸ್ಕೀಮಾ ರಚಿಸಲು
- ಪರಿಮಾಣ ಪರಿಶೀಲನೆ ನಿರ್ವಹಿಸಲು
- ಏಜೆಂಟ್‌ಗಳ ಸ್ವಯಂ ಕರೆದೊಯ್ಯಿಕೆಗೆ ಸೌಲಭ್ಯ ನೀಡಲು

ಈ ಡೆಮೋಗಾಗಿ:
- **ಸ್ಟಾಕ್‌ಹೋಮ್, ಸಿಯಾಟಲ್, ಟೋಕಿಯೋ, ಲಂಡನ್, ಆಂಸ್ಟರ್ಡ್ಯಾಮ್** → ಕೋಣೆಗಳಿವೆ ✅
- **ಇತರ ಎಲ್ಲಾ ನಗರಗಳು** → ಕೋಣೆಗಳಿಲ್ಲ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ಹಂತ 3: ಮಾರ್ಗ ನಿರ್ಣಯಕ್ಕಾಗಿ ಶರತ್ತು ಕಾರ್ಯಗಳನ್ನು ನಿರ್ಧರಿಸಿ

ಈ ಕಾರ್ಯಗಳು ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸಿ, ಕಾರ್ಯಪ್ರವಾಹದಲ್ಲಿ ಯಾವ ಮಾರ್ಗವನ್ನು ತೆಗೆದುಕೊಳ್ಳಬೇಕೆಂದು ನಿಶ್ಚಯಿಸುತ್ತವೆ.

**ಮುಖ್ಯ ಹಿನ್ನಲೆ:**
1. ಸಂದೇಶವು `AgentExecutorResponse` ಆಗಿದೆಯೇ ಎಂದು ಪರಿಶೀಲಿಸಿ
2. ರಚನೆಗೊಳಿಸಲಾದ ಔಟ್‌ಪುಟ್ (ಪಿಡ್ಯಾನ್ಟಿಕ್ ಮಾದರಿ) ಅನ್ನು ಪಾರ್ಸ್ ಮಾಡಿ
3. ಮಾರ್ಗದರ್ಶನವನ್ನು ನಿಯಂತ್ರಿಸಲು `True` ಅಥವಾ `False` ಅನ್ನು ಹಿಂತಿರುಗಿಸಿ

ಕಾರ್ಯಪ್ರವಾಹವು ಯಾವ ಎಕ್ಸಿಕ್ಯೂಟರ್‌ನ್ನು ನಂತರ ಕರೆಸಿಕೊಳ್ಳುವುದು ಎಂದು ನಿರ್ಧರಿಸಲು ಇವುಗಳನ್ನು **ಎಡ್‌ಜ್‌ಗಳು** ಮೇಲೆ ಮೌಲ್ಯಮಾಪನ ಮಾಡುತ್ತದೆ.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ಹಂತ 4: ಕಸ್ಟಮ್ ಡಿಸ್ಪ್ಲೇ ಎಂಬುದನ್ನು ರಚಿಸಿ

ಕಾರ್ಯಗತಗೊಳಿಸುವವರು ಪರಿವರ್ತನೆಗಳು ಅಥವಾ ಉಪಪ್ರಭಾವಗಳನ್ನು ನಡೆಸುವ ವರ್ಕ್‌ಫ್ಲೋ ಘಟಕಗಳಾಗಿದ್ದಾರೆ. ಸ್ಥೂಲ ಅಂತಿಮ ಫಲಿತಾಂಶವನ್ನು ಪ್ರದರ್ಶಿಸುವ ಕಸ್ಟಮ್ ಕಾರ್ಯಗತಗೊಳ್ಳುವವವನ್ನು ಸೃಷ್ಟಿಸಲು ನಾವು `@executor` ಡೆಕೋರೇಟರ್ ಅನ್ನು ಬಳಸುತ್ತೇವೆ.

**ಮುಖ್ಯ ಕಲ್ಪನೆಗಳು:**
- `@executor(id="...")` - ಫಂಕ್ಷನ್ ಅನ್ನು ವರ್ಕ್‌ಫ್ಲೋ ಕಾರ್ಯಗತಗೊಳಿಸುವವ તરીકે ನೋಂದಣಿ ಮಾಡುತ್ತದೆ
- `WorkflowContext[Never, str]` - ನಮೂದು/ಔಟ್‌ಪುಟ್ ためದ ಪ್ರಕಾರ ಸೂಚನೆಗಳು
- `ctx.yield_output(...)` - ಅಂತಿಮ ವರ್ಕ್‌ಫ್ಲೋ ಫಲಿತಾಂಶವನ್ನು ಒದಗಿಸುತ್ತದೆ


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ಹಂತ 5: ಪರಿಸರ ಚರಗಳನ್ನು ಲೋಡ್ ಮಾಡಿ

LLM ಗ್ರಾಹಕನನ್ನು ಕಾನ್ಫಿಗರ್ ಮಾಡಿ. ಈ ಉದಾಹರಣೆಗಳು ಈ ನಿಯಮಗಳೊಂದಿಗೆ ಕೆಲಸ ಮಾಡುತ್ತವೆ:
- **Microsoft Foundry** / **Azure OpenAI** (ಪ್ರತಿಕ್ರಿಯೆಗಳು API — ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ಹಂತ 6: ಸರಣೀಕೃತ output ಗಳೊಂದಿಗೆ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ಸೃಷ್ಟಿಸಿ

ನಾವು **ಮೂರು ಪರಿಣತಿ ಹೊಂದಿರುವ ಏಜೆಂಟ್‌ಗಳು** ಸೃಷ್ಟಿಸುತ್ತೇವೆ, ಪ್ರತಿಯೊಂದು `AgentExecutor` ನಲ್ಲಿ ಅಚ್ಚುಮೆಚ್ಚಾಗಿ ಲಪಟ್ಟಿರಿಸಲಾಗಿದೆ:

1. **availability_agent** - ಉಪಕರಣವನ್ನು ಉಪಯೋಗಿಸಿ ಹೋಟೆಲ್ ಲಭ್ಯತೆಯನ್ನು ಪರಿಶೀಲಿಸುತ್ತದೆ
2. **alternative_agent** - ಪರ್ಯಾಯ ನಗರಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ (ಕೊಠಡಿಗಳು ಇಲ್ಲದಿರುವಾಗ)
3. **booking_agent** - ಬುಕ್ಕಿಂಗ್‌ಗೆ ಪ್ರೋತ್ಸಾಹಿಸುತ್ತದೆ (ಕೊಠಡಿಗಳು ಲಭ್ಯವಿರುವಾಗ)

**ಪ್ರಧಾನ ವೈಶಿಷ್ಟ್ಯಗಳು:**
- `tools=[hotel_booking]` - ಏಜೆಂಟ್‌ಗಳಿಗೆ ಉಪಕರಣವನ್ನು ಒದಗಿಸುತ್ತದೆ
- `response_format=PydanticModel` - ಸರಣೀಕೃತ JSON output ಅನ್ನು ಬಲಪಡಿಸುತ್ತದೆ
- `AgentExecutor(..., id="...")` - ಕೆಲಸದ ಹಂತದಿಗಾಗಿ ಏಜೆಂಟ್ ಅನ್ನು ಲಟ್ಟಿಸುತ್ತಿದೆ


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ಹಂತ 7: ಶರತಗತ ಎಡ್ಜ್‌ಗಳೊಂದಿಗೆ ವರ್ಕ್‌ಫ್ಲೋ ನಿರ್ಮಿಸಿ

ಈಗ ನಾವು `WorkflowBuilder` ಅನ್ನು ಶರತಗತ ಮಾರ್ಗದರ್ಶನದೊಂದಿಗೆ ಗ್ರಾಫ್ ನಿರ್ಮಿಸಲು ಬಳಸುತ್ತೇವೆ:

**ವರ್ಕ್ಫ್ಲೋ ರಚನೆ:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ಪ್ರಮುಖ ವಿಧಾನಗಳು:**
- `.set_start_executor(...)` - ಪ್ರವೇಶ ಬಿಂದುವನ್ನು ಸೆಟ್ ಮಾಡುತ್ತದೆ
- `.add_edge(from, to, condition=...)` - ಶರತಗತ ಎಡ್ಜ್ ಅನ್ನು ಸೇರಿಸುತ್ತದೆ
- `.build()` - ವರ್ಕ್‌ಫ್ಲೋ ಅನ್ನು ಅಂತಿಮಗೊಳಿಸುತ್ತದೆ


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ಹಂತ 8: ಟೆಸ್ಟ್ ಪ್ರಕರಣ 1 ಚಲಾಯಿಸಿ - ನಗರ ಲಭ್ಯತೆ ಇಲ್ಲದೆ (ಪ್ಯಾರಿಸ್)

ಪ್ಯಾರಿಸ್‌ನಲ್ಲಿನ ಹೋಟೆಲ್‌ಗಳಿಗಾಗಿ ವಿನಂತಿ ಸಲ್ಲಿಸುವ ಮೂಲಕ **ಲಭ್ಯತೆ ಇಲ್ಲದ** ಮಾರ್ಗವನ್ನು ಪರೀಕ್ಷಿಸೋಣ (ನಮ್ಮ ಅನುಕರಣದಲ್ಲಿ ಇಲ್ಲಿ ಕೊಠಡಿಗಳು ಇಲ್ಲ).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ಹಂತ 9: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 2 ಓಡಿಸಿ - ನಗರ WITH ಲಭ್ಯತೆ (ಸ್ಟಾಕ್‌ಹೋಮ್)

ಈಗ ನಾವು **ಲಭ್ಯತೆ** ಮಾರ್ಗವನ್ನು ಸ್ಟಾಕ್‌ಹೋಮ್ನಲ್ಲಿರುವ ಹೋಟೆಲ್‌ಗಳನ್ನು ವಿನಂತಿಸುವ ಮೂಲಕ ಪರೀಕ್ಷಿಸೋಣ (ನಮ್ಮ ಸಿಂಪ್ಯುಲೇಶನ್‌ನಲ್ಲಿ ಕೊಠಡಿಗಳು ಲಭ್ಯವಿವೆ).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ಪ್ರಮುಖ ಅಂಶಗಳು ಮತ್ತು ಮುಂದಿನ ಹಂತಗಳು

### ✅ ನೀವು ಕಲಿತದ್ದು:

1. **WorkflowBuilder ಮಾದರಿ**
   - `.set_start_executor()` ಬಳಸಿ ಪ್ರವೇಶ ಬಿಂದುವನ್ನು ನಿರ್ಧರಿಸಿ
   - ಶರತ್ತು ಆಧಾರಿತ ಮಾರ್ಗದರ್ಶನಕ್ಕೆ `.add_edge(from, to, condition=...)` ಬಳಸಿ
   - ಕಾರ್ಯಪ್ರವಾಹವನ್ನು ಸ್ಥಿರಗೊಳಿಸಲು `.build()` ಅನ್ನು ಕರೆ ಮಾಡಿ

2. **ಶರತ್ತು ಆಧಾರಿತ ಮಾರ್ಗದರ್ಶನ**
   - ಶರತ್ತು ಕಾರ್ಯಗಳು `AgentExecutorResponse` ಅನ್ನು ಪರಿಶೀಲಿಸುತ್ತವೆ
   - ಮಾರ್ಗದರ್ಶನ ನಿರ್ಧಾರಗಳಿಗಾಗಿ ರಚಿತ ನಿರ್ಗಮებს ವಿಶ್ಲೇಷಿಸಿ
   - ಮಾರ್ಗವನ್ನು ಸಕ್ರಿಯಗೊಳಿಸಲು `True` ಮತ್ತು ತಪ್ಪಿಸುವುದಕ್ಕೆ `False` ಅನ್ನು ಹಿಂತಿರುಗಿಸಿ

3. **ಟೂಲ್ ಸಮನ್ವಯಿಕರಣ**
   - ಪೈಥಾನ್ ಕಾರ್ಯಗಳನ್ನು AI ಉಪಕರಣಗಳಾಗಿ ಪರಿವರ್ತಿಸಲು `@ai_function` ಬಳಸಿ
   - ಅಗತ್ಯವಾಗಿರುವಾಗ ಏಜೆಂಟ್ಗಳು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಉಪಕರಣಗಳನ್ನು ಕರೆಸಿಕೊಳ್ಳುತ್ತವೆ
   - ಉಪಕರಣಗಳು JSON ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತವೆ, ಅದನ್ನು ಏಜೆಂಟ್ಗಳು ವಿಶ್ಲೇಷಿಸುತ್ತವೆ

4. **ರಚಿತ ನಿರ್ಗಮಗಳು**
   - ಟೈಪ್-ಸುರಕ್ಷಿತ ಡೇಟಾ ಎಕ್ಸ್‌ಟ್ರ್ಯಾಕ್ಷನ್ ಸಮರ್ಥ ಪ್ಯಾಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಬಳಸಿ
   - ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವಾಗ `response_format=MyModel` ಅನ್ನು ಸೆಟ್ ಮಾಡಿ
   - ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು `Model.model_validate_json()` ಬಳಸಿ ವಿಶ್ಲೇಷಿಸಿ

5. **ಕಸ್ಟಮ್ ಆ执行ಕಾರರು**
   - ಕಾರ್ಯಪ್ರವಾಹ ಭಾಗಗಳನ್ನು ರಚಿಸಲು `@executor(id="...")` ಬಳಸಿ
   - ಕಾರ್ಯ ನಿರ್ವಹಕರು ಡೇಟಾವನ್ನು ಪರಿವರ್ತಿಸಲು ಅಥವಾ ಪಾರ್ಶ್ವ ಪರಿಣಾಮಗಳನ್ನು ಮಾಡುವುದಕ್ಕೆ ಸಾಧ್ಯವಿದೆ
   - ಕಾರ್ಯಪ್ರವಾಹ ಫಲಿತಾಂಶಗಳನ್ನು ಉತ್ಪಾದಿಸಲು `ctx.yield_output()` ಬಳಸಿ

### 🚀 ವಾಸ್ತವಿಕ ಜಗತ್ತಿನ ಅನ್ವಯಿಕೆಗಳು:

- **ಪ್ರಯಾಣ ಬುಕ್ಕಿಂಗ್**: ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸಿ, ಪರ್ಯಾಯಗಳನ್ನು ಸೂಚಿಸಿ, ಆಯ್ಕೆಯನ್ನು ಹೋಲಿಸಿ
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಸಮಸ್ಯೆಯ ಪ್ರಕಾರ, ಭಾವನೆ, ಆದ್ಯತೆಯ ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗದರ್ಶನ
- **ಇ-ಕಾಮರ್ಸ್**: ಇನ್ವೆಂಟರಿ ಪರಿಶೀಲಿಸಿ, ಪರ್ಯಾಯಗಳನ್ನ ಸೂಚಿಸಿ, ಆದೇಶಗಳನ್ನು ಪ್ರಕ್ರಿಯೆ ಮಾಡಿ
- **ವಿಷಯ ನಿಯಂತ್ರಣ**: ವಿಷಾಕ್ತತೆ ಅಂಕೆ, ಬಳಕೆದಾರ ಸೂಚನೆಗಳ ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗದರ್ಶನ
- **ಮಂಜೂರು ಕಾರ್ಯಪ್ರವಾಹಗಳು**: ಮೊತ್ತ, ಬಳಕೆದಾರ ಪಾತ್ರ, ಅಪಾಯ ಮಟ್ಟ ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗದರ್ಶನ
- **ಬಹು ಹಂತ ಪ್ರಕ್ರಿಯೆಗೊಳಿಸುವಿಕೆ**: ಡೇಟಾ ಗುಣಮಟ್ಟ, ಪೂರ್ಣತೆ ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗದರ್ಶನ

### 📚 ಮುಂದಿನ ಹಂತಗಳು:

- ಹೆಚ್ಚಿನ ಸಂಕೀರ್ಣ ಶರತ್ತುಗಳನ್ನು ಸೇರಿಸಿ (ಬಹು ಮಾನದಂಡಗಳು)
- ಕಾರ್ಯಪ್ರವಾಹ ಸ್ಥಿತಿ ನಿರ್ವಹಣೆಯೊಂದಿಗೆ ಲೂಪುಗಳನ್ನು ಜಾರಿಗೆ ತರುವುದಾಗಿ
- ಪುನರುಪಯೋಗಕ್ಕೆ ಅನುಕೂಲವಾಗುವ ಉಪ-ಕಾರ್ಯಪ್ರವಾಹಗಳನ್ನು ಸೇರಿಸಿ
- ವಾಸ್ತವ API ಗಳೊಡನೆ ಸಂಯೋಜಿಸಿ (ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್, ಇನ್ವೆಂಟರಿ ವ್ಯವಸ್ಥೆಗಳು)
- ದೋಷ ನಿರ್ವಹಣೆ ಮತ್ತು ಬ್ಯಾಕ್ಅಪ್ ಮಾರ್ಗಗಳನ್ನು ಸೇರಿಸಿ
- ಒಳಗೊಂಡಿರುವ ದೃಶ್ಯೀಕರಣ ಉಪಕರಣಗಳೊಂದಿಗೆ ಕಾರ್ಯಪ್ರವಾಹಗಳನ್ನು ದೃಶ್ಯೀಕರಿಸಿ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
